# Model 1: disruption forecast — run notebook
Run top to bottom. Every cell is safe to re-run.

1. **Setup**: paths and imports
2. **Load + label check**: confirms the data loaded correctly
3. **Fill `disruption_prob`**: writes `outputs/submission.csv`
4. **Submission check**: the doc's pre-upload checklist, all columns
5. **Files for Model 2 / Model 3**: hazards, duration curves, probabilities
6. **Results**: backtest by horizon, reliability, ablation chart (saved as PNGs for the video)
7. **Phase 2**: does nothing until weeks after 208 are in `Data/`

## 1. Setup

In [0]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import model1

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

# this notebook lives in Code Base/pipelining/Model 1/iter3; Data/ is at the repo root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../../.."))
DATA = os.path.join(REPO_ROOT, "Data")
if not os.path.isdir(DATA):
    DATA = "/Workspace/Repos/DSDG Hackathon/DSDG Hackathon/Data"
OUT = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT, exist_ok=True)

print("DATA:", DATA)
print("OUT: ", OUT)
print(sorted(os.listdir(DATA)))

## 2. Load data and check labels
Both off-diagonal cells must be 0: the shock sources rebuilt from supply_history / tariffs / shipping match `disrupted` exactly.

In [0]:
panels = model1.Panels(model1.load_data(DATA))
PHASE2 = panels.last_week > 208
print(f"features run through week {panels.last_week}, outcomes through week {panels.last_outcome_week}")
print("Phase 2 data loaded:", PHASE2)

labels = model1.check_labels(panels)
print(labels)
assert labels.loc[0, 1] == 0 and labels.loc[1, 0] == 0, "labels don't rebuild - check the CSVs in Data"


## 3. Fill `disruption_prob`
Reads the team's `outputs/submission.csv` if it exists (keeps Model 2 / Model 3 columns), otherwise starts from the sample.
Before Phase 2 data, weeks 222-234 are forecast from week 208 (14-26 weeks ahead = background rate only).

In [0]:
SUB_PATH = os.path.join(OUT, "submission.csv")
sub_in = SUB_PATH if os.path.exists(SUB_PATH) else os.path.join(DATA, "sample_submission.csv")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    sub = model1.fill_submission(panels, sub_in, out_path=SUB_PATH)

print("read:  ", sub_in)
print("wrote: ", SUB_PATH)
if not PHASE2:
    print("note: weeks 222-234 use the week-208 origin until Phase 2 data is loaded")

wk = sub.id.str.extract(r"^w(\d+)_(.+)$")
probs = sub.assign(week=wk[0].astype(int), material=wk[1])
probs.pivot(index="week", columns="material", values="disruption_prob").round(3)

## 4. Submission check (all columns)
Run this before every upload. Model 2 / Model 3 columns are checked too once they are filled.

In [0]:
def check_submission(sub, suppliers):
    issues = []
    if len(sub) != 260:
        issues.append(f"{len(sub)} rows, expected 260")
    if sub.id.duplicated().any():
        issues.append("duplicate ids")
    if sub.isna().any().any():
        issues.append(f"missing values in {sub.columns[sub.isna().any()].tolist()}")
    if not sub.disruption_prob.between(0, 1).all():
        issues.append("disruption_prob outside [0, 1]")
    if not (sub.price_forecast > 0).all():
        issues.append("price_forecast <= 0")
    if (sub.order_units < 0).any() or not np.allclose(sub.order_units, sub.order_units.round()):
        issues.append("order_units negative or not whole numbers")
    share_cols = [c for c in sub.columns if c.startswith("share_")]
    if not np.allclose(sub[share_cols].sum(axis=1), 1.0, atol=1e-6):
        issues.append("shares don't sum to 1 on every row")
    mat = sub.id.str.extract(r"^w\d+_(.+)$")[0]
    sells = suppliers.groupby("material").country.apply(set)
    for col in share_cols:
        cty = col.split("_", 1)[1]
        bad = (sub[col] > 0) & ~mat.map(lambda m: cty in sells[m])
        if bad.any():
            issues.append(f"{col} > 0 for materials {cty} doesn't sell: {sorted(mat[bad].unique())}")
    return issues

problems = check_submission(sub, panels.suppliers)
print("READY TO UPLOAD" if not problems else "FIX BEFORE UPLOAD:\n- " + "\n- ".join(problems))

## 5. Files for Model 2 and Model 3
- `disruption_probs.csv`: long format, first window (Model 2's shock-jump rule uses p > 0.5). For both windows read `disruption_prob` from `submission.csv`.
- `hazards.csv`: per supplier per week: `q_*` = chance a shock of that kind starts that week, `p_*` = chance one is active.
- `hazards_sources.csv`: lanes and countries. Draw each once per scenario and apply it to every supplier on that lane / in that country.
- `duration_curves.csv`: P(shock lasts at least x weeks) by type.

In [0]:
m = model1.DisruptionModel().fit(panels)
sup_hz, src_hz = m.hazards()
sup_hz.to_csv(os.path.join(OUT, "hazards.csv"), index=False)
src_hz.to_csv(os.path.join(OUT, "hazards_sources.csv"), index=False)
m.duration.curves().to_csv(os.path.join(OUT, "duration_curves.csv"), index=False)
m.predict().to_csv(os.path.join(OUT, "disruption_probs.csv"), index=False)
print("forecast origin: week", m.origin)

active = sup_hz[(sup_hz.h == 1) & (sup_hz[["state_supplier", "state_country", "state_lane", "state_tariff"]] != "").any(axis=1)]
print("\nshocks active at the origin:", "none" if active.empty else "")
if not active.empty:
    print(active[["material", "country", "state_supplier", "state_country", "state_lane", "state_tariff"]].to_string(index=False))

print("\nhighest-risk suppliers 4 weeks out:")
print(sup_hz[sup_hz.h == 4].nlargest(8, "p_any")[["material", "country", "lane", "p_supplier", "p_country", "p_lane", "p_tariff", "p_any"]].round(3).to_string(index=False))

## 6. Results
Rolling backtest: forecast from every 4th week (104-204), 1-13 weeks ahead, using only data known at the time.
Skill = 1 - Brier / Brier of the base-rate baseline (0 = baseline, 1 = perfect).

In [0]:
bt = model1.backtest(panels)
print(bt.summary().round(3).to_string())

def skill_by(rows, key):
    r = rows.assign(se=(rows.p - rows.y) ** 2, sb=(rows.baseline - rows.y) ** 2)
    g = r.groupby(key).agg(brier=("se", "mean"), brier_base=("sb", "mean"), n=("y", "size"), events=("y", "sum"))
    g["skill"] = 1 - g.brier / g.brier_base
    return g

by_h = skill_by(bt.rows, "h")
by_mat = skill_by(bt.rows, "material")
rel = bt.reliability("p")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(by_h.index, by_h.skill, color="#4C72B0")
ax[0].axhline(0, color="black", lw=0.8)
ax[0].set(title="Brier skill by weeks ahead", xlabel="weeks ahead (h)", ylabel="skill vs base rate")
ax[1].plot([0, 1], [0, 1], "--", color="grey", lw=1)
ax[1].scatter(rel.mean_pred, rel.actual_rate, s=np.sqrt(rel.n) * 4, color="#4C72B0")
ax[1].set(title="Reliability (dot size = rows)", xlabel="predicted probability", ylabel="actual disruption rate", xlim=(0, 1), ylim=(0, 1))
plt.tight_layout()
fig.savefig(os.path.join(OUT, "backtest_horizon_reliability.png"), dpi=150)
plt.show()

print(by_mat.round(3).sort_values("skill").to_string())

**Ablation ladder** (the doc's v0-v4). Slow: it re-runs the backtest 5 times. Loads the saved table unless `RERUN_ABLATION = True`.

In [0]:
RERUN_ABLATION = False
ab_path = os.path.join(OUT, "ablation.csv")
if RERUN_ABLATION or not os.path.exists(ab_path):
    ab = model1.ablation(panels)
    ab.to_csv(ab_path)
else:
    ab = pd.read_csv(ab_path, index_col=0)

cols = ["overall", "calm_at_origin", "in_shock_at_origin", "h1-4", "h5-8", "h9-13", "doc_origins"]
display_ab = ab[[c for c in cols if c in ab.columns]].round(3)
print(display_ab.to_string())

plot_ab = ab.rename(index={"final + Platt calibration": "+ Platt calibration (not used)"})
used = plot_ab.drop(index="+ Platt calibration (not used)").overall.idxmax()   # the version the submission uses
colors = ["#DD8452" if name == used else "#bbbbbb" for name in plot_ab.index]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(plot_ab.index[::-1], plot_ab.overall[::-1], color=colors[::-1])
for i, v in enumerate(plot_ab.overall[::-1]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center")
ax.set(title="Model 1 ablation: Brier skill vs base rate (orange = submitted)",
       xlabel="skill (0 = base rate)", xlim=(0, plot_ab.overall.max() * 1.15))
ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
fig.savefig(os.path.join(OUT, "ablation.png"), dpi=150)
plt.show()

## 7. Phase 2
Does nothing until `Data/` has weeks after 208. Then: checks which signals went past their training range (the early sign of the big shock), re-scores, and refills the submission from week 221. Re-run sections 2-6 afterwards and send the new files to Model 2 / Model 3.

If Phase 2 arrives as separate files with only the new weeks, append them first (set `PHASE2_DIR` and uncomment).

In [0]:
# PHASE2_DIR = "/Workspace/..."   # folder with the Phase 2 CSVs, if they only contain the new weeks
# for name in ["outcomes", "supply_history", "tariffs", "prices", "shipping", "macro", "news", "demand"]:
#     old = pd.read_csv(os.path.join(DATA, f"{name}.csv"))
#     new = pd.read_csv(os.path.join(PHASE2_DIR, f"{name}.csv"))
#     pd.concat([old, new]).drop_duplicates().to_csv(os.path.join(DATA, f"{name}.csv"), index=False)

panels = model1.Panels(model1.load_data(DATA))
if panels.last_week <= 208:
    print(f"No Phase 2 data yet (features end at week {panels.last_week}). Nothing to do.")
else:
    print(model1.check_labels(panels))
    ext = model1.signal_extremes(panels, train_end=208, recent_start=209)
    print("\nsignals past their training max:")
    print(ext[ext.exceeds].head(20).to_string(index=False) if ext.exceeds.any() else "none")
    print(model1.backtest(panels).summary().round(3).to_string())
    sub = model1.fill_submission(panels, SUB_PATH, out_path=SUB_PATH)
    print("refilled", SUB_PATH, "from origin week", min(221, panels.last_week))